# Linear Elasticity

![](linear_elasticity.svg)

*Figure 1*: Linear elastically deformed 1mm $\times$ 1mm Ferrite logo.



## Implementation

The following code is based on the [Linear Elasticity](https://ferrite-fem.github.io/Ferrite.jl/stable/tutorials/linear_elasticity/) tutorial from the Ferrite.jl documentation, with some comments removed for brevity.
There are two main modifications:

1. Fourth-order `Lagrange` shape functions are used for field approximation: `ip = Lagrange{RefTriangle,4}()^2`.
2. High-order quadrature points are used to accommodate the fourth-order shape functions: `qr = QuadratureRule{RefTriangle}(8)`.

In [1]:
using Ferrite, FerriteGmsh, FerriteOperators, FerriteMultigrid, AlgebraicMultigrid
using Downloads: download
using IterativeSolvers
using TimerOutputs

TimerOutputs.enable_debug_timings(AlgebraicMultigrid)
TimerOutputs.enable_debug_timings(FerriteMultigrid)

Emod = 200.0e3 # Young's modulus [MPa]
ν = 0.3        # Poisson's ratio [-]

Gmod = Emod / (2(1 + ν))  # Shear modulus
Kmod = Emod / (3(1 - 2ν)) # Bulk modulus

C = gradient(ϵ -> 2 * Gmod * dev(ϵ) + 3 * Kmod * vol(ϵ), zero(SymmetricTensor{2,2}))

function assemble_external_forces!(f_ext, dh, facetset, facetvalues, prescribed_traction)
    # Create a temporary array for the facet's local contributions to the external force vector
    fe_ext = zeros(getnbasefunctions(facetvalues))
    for facet in FacetIterator(dh, facetset)
        # Update the facetvalues to the correct facet number
        reinit!(facetvalues, facet)
        # Reset the temporary array for the next facet
        fill!(fe_ext, 0.0)
        # Access the cell's coordinates
        cell_coordinates = getcoordinates(facet)
        for qp in 1:getnquadpoints(facetvalues)
            # Calculate the global coordinate of the quadrature point.
            x = spatial_coordinate(facetvalues, qp, cell_coordinates)
            tₚ = prescribed_traction(x)
            # Get the integration weight for the current quadrature point.
            dΓ = getdetJdV(facetvalues, qp)
            for i in 1:getnbasefunctions(facetvalues)
                Nᵢ = shape_value(facetvalues, qp, i)
                fe_ext[i] += tₚ ⋅ Nᵢ * dΓ
            end
        end
        # Add the local contributions to the correct indices in the global external force vector
        assemble!(f_ext, celldofs(facet), fe_ext)
    end
    return f_ext
end

function assemble_cell!(ke, cellvalues, C)
    for q_point in 1:getnquadpoints(cellvalues)
        # Get the integration weight for the quadrature point
        dΩ = getdetJdV(cellvalues, q_point)
        for i in 1:getnbasefunctions(cellvalues)
            # Gradient of the test function
            ∇Nᵢ = shape_gradient(cellvalues, q_point, i)
            for j in 1:getnbasefunctions(cellvalues)
                # Symmetric gradient of the trial function
                ∇ˢʸᵐNⱼ = shape_symmetric_gradient(cellvalues, q_point, j)
                ke[i, j] += (∇Nᵢ ⊡ C ⊡ ∇ˢʸᵐNⱼ) * dΩ
            end
        end
    end
    return ke
end

function assemble_global!(K, dh, cellvalues, C)
    # Allocate the element stiffness matrix
    n_basefuncs = getnbasefunctions(cellvalues)
    ke = zeros(n_basefuncs, n_basefuncs)
    # Create an assembler
    assembler = start_assemble(K)
    # Loop over all cells
    for cell in CellIterator(dh)
        # Update the shape function gradients based on the cell coordinates
        reinit!(cellvalues, cell)
        # Reset the element stiffness matrix
        fill!(ke, 0.0)
        # Compute element contribution
        assemble_cell!(ke, cellvalues, C)
        # Assemble ke into K
        assemble!(assembler, celldofs(cell), ke)
    end
    return K
end

function linear_elasticity_2d(C)
    logo_mesh = "logo.geo"
    asset_url = "https://raw.githubusercontent.com/Ferrite-FEM/Ferrite.jl/gh-pages/assets/"
    isfile(logo_mesh) || download(string(asset_url, logo_mesh), logo_mesh)

    grid = togrid(logo_mesh)
    addfacetset!(grid, "top", x -> x[2] ≈ 1.0) # facets for which x[2] ≈ 1.0 for all nodes
    addfacetset!(grid, "left", x -> abs(x[1]) < 1.0e-6)
    addfacetset!(grid, "bottom", x -> abs(x[2]) < 1.0e-6)

    dim = 2
    order = 4
    ip = Lagrange{RefTriangle,order}()^dim # vector valued interpolation
    ip_coarse = Lagrange{RefTriangle,1}()^dim

    qr = QuadratureRule{RefTriangle}(8)
    qr_face = FacetQuadratureRule{RefTriangle}(6)

    cellvalues = CellValues(qr, ip)
    facetvalues = FacetValues(qr_face, ip)

    dhh = DofHandlerHierarchy(grid, 2)
    add!(dhh, :u, [ip_coarse, ip])
    close!(dhh)

    chh = ConstraintHandlerHierarchy(dhh)
    add!(chh, dh->Dirichlet(:u, getfacetset(dh.grid, "bottom"), (x, t) -> 0.0, 2))
    add!(chh, dh->Dirichlet(:u, getfacetset(dh.grid, "left"), (x, t) -> 0.0, 1))
    close!(chh)

    traction(x) = Vec(0.0, 20.0e3 * x[1])

    dh = dhh[end]
    ch = chh[end]

    A = allocate_matrix(dh)
    assemble_global!(A, dh, cellvalues, C)

    b = zeros(ndofs(dh))
    assemble_external_forces!(b, dh, getfacetset(grid, "top"), facetvalues, traction)
    apply!(A, b, ch)

    return A, b, dhh, chh
end

linear_elasticity_2d (generic function with 1 method)

### Rediscretization

For the rediscretization approach we can define the assembly for the levels via FerriteOperators directly.

In [2]:
"""
    LinearElasticityIntegrator{TC, QRC} <: AbstractBilinearIntegrator

Multigrid problem for linear elasticity.
Implements the FerriteOperators `AbstractBilinearIntegrator` interface.
"""
struct LinearElasticityIntegrator{TC <: SymmetricTensor, QRC} <: FerriteOperators.AbstractBilinearIntegrator
    ℂ::TC  # material stiffness tensor (4th order)
    qrc::QRC
end

struct LinearElasticityElementCache{CV, TC} <: FerriteOperators.AbstractVolumetricElementCache
    cv::CV
    ℂ::TC
end

function FerriteOperators.setup_element_cache(problem::LinearElasticityIntegrator, sdh::SubDofHandler)
    qr     = getquadraturerule(problem.qrc, sdh)
    ip     = Ferrite.getfieldinterpolation(sdh, first(Ferrite.getfieldnames(sdh)))
    first_cell = getcells(Ferrite.get_grid(sdh.dh), first(sdh.cellset))
    ip_geo = Ferrite.geometric_interpolation(typeof(first_cell))
    cv     = CellValues(qr, ip, ip_geo)
    return LinearElasticityElementCache(cv, problem.ℂ)
end

FerriteOperators.reinit_values!(cache::LinearElasticityElementCache, cell) = reinit!(cache.cv, cell)

FerriteOperators.provides_analytic(::Type{<:LinearElasticityElementCache}, ::JacobianKind) = true
function FerriteOperators.assemble_cell!(req::JacobianRequest{:u}, cache::LinearElasticityElementCache, args::CellArgs)
    ℂ = cache.ℂ
    for q_point in 1:getnquadpoints(cache.cv)
        dΩ = getdetJdV(cache.cv, q_point)
        for i in 1:getnbasefunctions(cache.cv)
            ∇ˢʸᵐNᵢ = shape_symmetric_gradient(cache.cv, q_point, i)
            for j in 1:getnbasefunctions(cache.cv)
                ∇ˢʸᵐNⱼ = shape_symmetric_gradient(cache.cv, q_point, j)
                req.K[i, j] += (∇ˢʸᵐNᵢ ⊡ ℂ ⊡ ∇ˢʸᵐNⱼ) * dΩ
            end
        end
    end
end

The bilinear form induces a linear operator, so its residual is the element
matrix acting on the element vector — mandatory so the element also
composes into nonlinear operators.

In [3]:
function FerriteOperators.assemble_cell!(req::ResidualRequest, cache::LinearElasticityElementCache, args::CellArgs)
    ℂ = cache.ℂ
    uₑ = args.states.u
    for q_point in 1:getnquadpoints(cache.cv)
        dΩ = getdetJdV(cache.cv, q_point)
        εu = function_symmetric_gradient(cache.cv, q_point, uₑ)
        for i in 1:getnbasefunctions(cache.cv)
            req.r[i] += (shape_symmetric_gradient(cache.cv, q_point, i) ⊡ ℂ ⊡ εu) * dΩ
        end
    end
end

### Near Null Space (NNS)

In multigrid methods for problems with vector-valued unknowns, such as linear elasticity,
the near null space represents the low energy mode or the smooth error that needs to be captured
in the coarser grid when using SA-AMG (Smoothed Aggregation Algebraic Multigrid), more on the topic
can be found  in [schroder2010](@citet).

For 2D linear elasticity problems, the rigid body modes are:
1. Translation in the x-direction,
2. Translation in the y-direction,
3. Rotation about the z-axis (i.e., $x_3$): each point (x, y) is mapped to (-y, x).

The function `create_nns` constructs the NNS matrix `B ∈ ℝ^{n × 3}`, where `n` is the number of degrees of freedom (DOFs)
for the case of `p` = 1 (i.e., linear interpolation), because `B` is only relevant for AMG.

In [4]:
function create_nns(dh, fieldname = first(dh.field_names))
    @assert length(dh.field_names) == 1 "Only a single field is supported for now."

    coords_flat = zeros(ndofs(dh))
    apply_analytical!(coords_flat, dh, fieldname, x -> x)
    coords = reshape(coords_flat, (length(coords_flat) ÷ 2, 2))

    grid = dh.grid
    B = zeros(Float64, ndofs(dh), 3)
    B[1:2:end, 1] .= 1 # x - translation
    B[2:2:end, 2] .= 1 # y - translation

    # in-plane rotation (x,y) → (-y,x)
    x = coords[:, 1]
    y = coords[:, 2]
    B[1:2:end, 3] .= -y
    B[2:2:end, 3] .= x

    return B
end

create_nns (generic function with 2 methods)

### Setup the linear elasticity problem
Load `FerriteMultigrid` to access the p-multigrid solver.

In [5]:
using FerriteMultigrid

Construct the linear elasticity problem with 4th order polynomial shape functions.

In [6]:
A, b, dhh, chh = linear_elasticity_2d(C);

Info    : Reading 'logo.geo'...
Info    : Done reading 'logo.geo'
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 20%] Meshing curve 4 (Line)
Info    : [ 30%] Meshing curve 5 (Line)
Info    : [ 30%] Meshing curve 6 (Line)
Info    : [ 40%] Meshing curve 7 (Line)
Info    : [ 40%] Meshing curve 8 (Line)
Info    : [ 50%] Meshing curve 9 (Line)
Info    : [ 60%] Meshing curve 10 (Line)
Info    : [ 60%] Meshing curve 11 (Line)
Info    : [ 70%] Meshing curve 12 (Line)
Info    : [ 70%] Meshing curve 13 (Line)
Info    : [ 80%] Meshing curve 14 (Line)
Info    : [ 80%] Meshing curve 15 (Line)
Info    : [ 90%] Meshing curve 16 (Line)
Info    : [ 90%] Meshing curve 17 (Line)
Info    : [100%] Meshing curve 18 (Line)
Info    : Done meshing 1D (Wall 0.00107873s, CPU 0.001079s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 2 (

Construct the near null space (NNS) matrix

In [7]:
B = create_nns(dhh[1])

208×3 Matrix{Float64}:
 1.0  0.0  -1.0
 0.0  1.0   0.801535
 1.0  0.0  -0.253172
 0.0  1.0   0.454964
 1.0  0.0  -0.612999
 0.0  1.0   0.900767
 1.0  0.0  -0.0858959
 0.0  1.0   0.417361
 1.0  0.0  -0.66085
 0.0  1.0   0.820318
 ⋮         
 0.0  1.0   0.11219
 1.0  0.0  -0.464069
 0.0  1.0   0.592985
 1.0  0.0  -0.450733
 0.0  1.0   0.301241
 1.0  0.0  -0.677392
 0.0  1.0   1.0
 1.0  0.0  -0.477795
 0.0  1.0   0.126586

> **Danger**
>
> Since NNS matrix is only relevant for AMG, and it is not used in the p-multigrid solver, therefore, `B` has to provided using linear field approximation (i.e., `p = 1`) when using AMG as the coarse solver, otherwise (e.g., using `Pinv` as the coarse solver), then we don't have to provide it.

### P-multigrid Configuration

In [8]:
reset_timer!()

pcoarse_solver = SmoothedAggregationCoarseSolver(; B)

SmoothedAggregationCoarseSolver{Tuple{}, Base.Pairs{Symbol, Matrix{Float64}, Nothing, @NamedTuple{B::Matrix{Float64}}}}((), Base.Pairs(:B => [1.0 0.0 -1.0; 0.0 1.0 0.801534880751; … ; 1.0 0.0 -0.47779490322723067; 0.0 1.0 0.1265859930873549]))

#### 0. CG as baseline

In [9]:
@timeit "CG" x_cg = IterativeSolvers.cg(A, b; maxiter = 1000, verbose=false)

2914-element Vector{Float64}:
 -0.009922773140937731
  0.027908992918940487
 -0.01227354490809059
  0.02778766287825086
 -0.011716098076385625
  0.03380248712536394
 -0.010480498289947952
  0.027964097065306594
 -0.011058650993832485
  0.02796370462681717
  ⋮
  0.020447374000925944
 -0.0034252013983498398
  0.019667499675877137
 -0.006499317727402122
  0.02796650153472074
 -0.0070905775885465655
  0.029186469235605944
 -0.007055602278683182
  0.02916954209841766

#### 1. Galerkin Coarsening Strategy

In [10]:
config_gal = pmultigrid_config(coarse_strategy = Galerkin())
@timeit "Galerkin only" x_gal, res_gal = solve(A, b, dhh, chh, config_gal; pcoarse_solver, log=true, maxiter = 1000, rtol = 1e-10)

builder_gal = PMultigridPreconBuilder(dhh, chh, config_gal; pcoarse_solver)
@timeit "Build preconditioner" Pl_gal = builder_gal(A)[1]
@timeit "Galerkin CG" x_gcg, res_gcg = IterativeSolvers.cg(A, b; Pl = Pl_gal, maxiter = 1000, log=true, verbose=false)

([-0.009922773159831979, 0.027908992912073955, -0.012273544894648485, 0.02778766291151923, -0.011716098097380147, 0.033802487124505495, -0.010480498302216283, 0.027964097066534497, -0.011058650998156056, 0.027963704639539656  …  -0.003571423564662785, 0.020447373993602923, -0.0034252014257079868, 0.019667499670456293, -0.00649931771238124, 0.027966501509221774, -0.0070905775684436765, 0.02918646920821057, -0.007055602262622334, 0.02916954207108605], Converged after 25 iterations.)

#### 2. Rediscretization Coarsening Strategy

In [11]:
# Rediscretization Coarsening Strategy
config_red = pmultigrid_config(coarse_strategy = Rediscretization(LinearElasticityIntegrator(C, QuadratureRuleCollection(7))))
@timeit "Rediscretization only" x_red, res_red = solve(A, b, dhh, chh, config_red; pcoarse_solver, log=true, maxiter = 1000, rtol = 1e-10)

builder_red = PMultigridPreconBuilder(dhh, chh, config_red; pcoarse_solver)
@timeit "Build preconditioner" Pl_red = builder_red(A)[1]
@timeit "Rediscretization CG" x_rcg, res_rcg = IterativeSolvers.cg(A, b; Pl = Pl_red, maxiter = 1000, log=true, verbose=false)

print_timer(title = "Analysis with $(getncells(dhh[end].grid)) elements", linechars = :ascii)

-----------------------------------------------------------------------------------------------
         Analysis with 174 elements                   Time                    Allocations      
                                             -----------------------   ------------------------
              Tot / % measured:                   4.56s /  82.8%            395MiB /  80.5%    

Section                              ncalls     time    %tot     avg     alloc    %tot      avg
-----------------------------------------------------------------------------------------------
Rediscretization only                     1    1.68s   44.5%   1.68s    197MiB   61.9%   197MiB
  init                                    1    691ms   18.3%   691ms   61.3MiB   19.2%  61.3MiB
    pmultigrid numeric                    1    690ms   18.3%   690ms   60.0MiB   18.8%  60.0MiB
      setup coarse operator               1    451ms   11.9%   451ms   45.7MiB   14.4%  45.7MiB
      assemble coarse operator         

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*